# Exploring the BigQuery OpenAQ (Air Quality) dataset with SQL queries

## Setup

In [ ]:
PROJECT_ID = 'sql-project-openaq'

In [1]:
from google.cloud import bigquery
client = bigquery.Client(project=PROJECT_ID)
dataset = client.get_dataset("bigquery-public-data.openaq")

Connected


### Print the list of tables available in the OpenAQ dataset

In [3]:
tables = list(client.list_tables(dataset))
print([table.table_id for table in tables])

['global_air_quality']


### Have a look at the schema to get variable names and data types

In [5]:
table = client.get_table("bigquery-public-data.openaq.global_air_quality")
table.schema

[SchemaField('location', 'STRING', 'NULLABLE', None, None, (), None, None),
 SchemaField('city', 'STRING', 'NULLABLE', None, None, (), None, None),
 SchemaField('country', 'STRING', 'NULLABLE', None, None, (), None, None),
 SchemaField('pollutant', 'STRING', 'NULLABLE', None, None, (), None, None),
 SchemaField('value', 'FLOAT', 'NULLABLE', None, None, (), None, None),
 SchemaField('timestamp', 'TIMESTAMP', 'NULLABLE', None, None, (), None, None),
 SchemaField('unit', 'STRING', 'NULLABLE', None, None, (), None, None),
 SchemaField('source_name', 'STRING', 'NULLABLE', None, None, (), None, None),
 SchemaField('latitude', 'FLOAT', 'NULLABLE', None, None, (), None, None),
 SchemaField('longitude', 'FLOAT', 'NULLABLE', None, None, (), None, None),
 SchemaField('averaged_over_in_hours', 'FLOAT', 'NULLABLE', None, None, (), None, None),
 SchemaField('location_geom', 'GEOGRAPHY', 'NULLABLE', None, None, (), None, None)]

In [6]:
client.list_rows(table, max_results=5).to_dataframe()

,location,city,country,pollutant,value,timestamp,unit,source_name,latitude,longitude,averaged_over_in_hours,location_geom
0,"Borówiec, ul. Drapałka",Borówiec,PL,bc,0.85217,2022-04-28 07:00:00+00:00,µg/m³,GIOS,1.0,52.276794,17.074114,POINT(52.276794 1)
1,"Kraków, ul. Bulwarowa",Kraków,PL,bc,0.91284,2022-04-27 23:00:00+00:00,µg/m³,GIOS,1.0,50.069308,20.053492,POINT(50.069308 1)
2,"Płock, ul. Reja",Płock,PL,bc,1.41000,2022-03-30 04:00:00+00:00,µg/m³,GIOS,1.0,52.550938,19.709791,POINT(52.550938 1)
3,"Elbląg, ul. Bażyńskiego",Elbląg,PL,bc,0.33607,2022-05-03 13:00:00+00:00,µg/m³,GIOS,1.0,54.167847,19.410942,POINT(54.167847 1)
4,"Piastów, ul. Pułaskiego",Piastów,PL,bc,0.51000,2022-05-11 05:00:00+00:00,µg/m³,GIOS,1.0,52.191728,20.837489,POINT(52.191728 1)


### Check a few value ranges
Let's have a look at a few values from the data: minimum and maximum recorded values for each country, years when data collection started and ended, number of measurements for each pollutant per country.

In [149]:
data_look_query = """
    SELECT country, pollutant,
            MIN(value) AS min_value, MAX(value) AS max_value,
            MIN(EXTRACT(YEAR from timestamp)) AS start_year, MAX(EXTRACT(YEAR from timestamp)) AS end_year,
            COUNT(1) AS num_values
    FROM `bigquery-public-data.openaq.global_air_quality`
    GROUP BY pollutant, country
    ORDER BY country
    """

ONE_GB = int(1e9)
safe_config = bigquery.QueryJobConfig(maximum_bytes_billed=ONE_GB)

data_look_query_job = client.query(data_look_query, job_config=safe_config)
data_look = data_look_query_job.to_dataframe()

print(data_look)

    country pollutant   min_value    max_value  start_year  end_year  \
0        AD       so2    0.000000     3.000000        2020      2022   
1        AD        co    0.000000   400.000000        2020      2022   
2        AD      pm10    0.000000   160.000000        2020      2022   
3        AD       no2    0.000000    57.000000        2020      2022   
4        AD        o3    4.000000   120.000000        2020      2022   
..      ...       ...         ...          ...         ...       ...   
395      ZA        co   -1.022000    61.745000        2020      2022   
396      ZA        o3   -0.028535     1.664363        2021      2022   
397      ZA      pm25 -212.138000  2356.532000        2020      2022   
398      ZA       so2   -0.731225     0.364986        2019      2022   
399      ZA      pm10 -273.409000  2357.143000        2019      2022   

     num_values  
0           424  
1           422  
2           435  
3           418  
4          1221  
..          ...  
395      

/usr/local/lib/python3.12/dist-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


Notice that some minimum values are negative. These are likely errors so in the queries below we discard negative pollutant values, where relevant.

## 1. Which countries have the most air quality monitoring stations included in the dataset?

In [24]:
Q1_query = """
    SELECT country, COUNT(DISTINCT location) as num_stations
    FROM `bigquery-public-data.openaq.global_air_quality`
    GROUP BY country
    ORDER BY num_stations DESC
    """
# Create a QueryJobConfig object to estimate size of query without running it
dry_run_config = bigquery.QueryJobConfig(dry_run=True)

# API request - dry run query to estimate costs
dry_run_query_job = client.query(Q1_query, job_config=dry_run_config)

print("This query will process {} Mb.".format(dry_run_query_job.total_bytes_processed/1e6))

This query will process 12.0695069 Mb.


In [25]:
Q1_query_job = client.query(Q1_query, job_config=safe_config)
Q1_result = Q1_query_job.to_dataframe()

print(Q1_result)

/usr/local/lib/python3.12/dist-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


    country  num_stations
0        US          2330
1        CN          1828
2        FR           660
3        ES           629
4        IT           577
..      ...           ...
106      ML             1
107      TJ             1
108      TD             1
109      CS             1
110      MG             1

[111 rows x 2 columns]


## 2. Which pollutants are tracked and how many readings exist for each?

In [21]:
Q2_query = """
    SELECT pollutant, COUNT(1) as num_readings
    FROM `bigquery-public-data.openaq.global_air_quality`
    GROUP BY pollutant
    ORDER BY num_readings DESC
    """

dry_run_query_job = client.query(Q2_query, job_config=dry_run_config)
print("This query will process {} Mb.".format(dry_run_query_job.total_bytes_processed/1e6))

This query will process 2.8417978 Mb.


In [23]:
Q2_query_job = client.query(Q2_query, job_config=safe_config)
Q2_result = Q2_query_job.to_dataframe()

print(Q2_result)

  pollutant  num_readings
0        o3       1273021
1      pm25       1200025
2       no2       1047525
3      pm10        990912
4       so2        610123
5        co        431979
6        bc         41028
7        no             1


/usr/local/lib/python3.12/dist-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


### Q2 Answer
The pollutants tracked are, by descending number of readings:
- Ozone (O$_3$) at ground level
- Fine particles (PM$_{2.5}$)
- Nitrogen dioxide (NO$_2$)
- Inhalable coarse particles (PM$_{10}$)
- Sulfur dioxide (SO$_2$)
- Carbon monoxide (CO)
- Black carbon (BC)
- Nitric oxide (NO)

## 3. Which cities have recorded at least 5 dangerously high PM$_{2.5}$ values (average above 50 µg/m³)?

In [46]:
Q3_query = """
    SELECT city, COUNT(1) as num_readings, AVG(value) as avg_pm25
    FROM `bigquery-public-data.openaq.global_air_quality`
    WHERE value > 50 AND pollutant = "pm25"
    GROUP BY city
    HAVING num_readings >= 5
    ORDER BY avg_pm25 DESC
    """

dry_run_query_job = client.query(Q3_query, job_config=dry_run_config)
print("This query will process {} Mb.".format(dry_run_query_job.total_bytes_processed/1e6))

This query will process 146.610998 Mb.


In [45]:
Q3_query_job = client.query(Q3_query, job_config=safe_config)
Q3_result = Q3_query_job.to_dataframe()

print(Q3_result)

                        city  num_readings    avg_pm25
0                      Delhi         11291  121.520568
1                     Mumbai          2389  202.750578
2                  Ahmedabad          2149  276.197636
3                        N/A          1894  125.142714
4                      Patna          1888   96.261430
..                       ...           ...         ...
387                     MONO             5   63.460000
388           Guatemala City             5   52.400000
389        Nakło nad Notecią             5   60.040000
390  Phoenix-Mesa-Scottsdale             5   87.020000
391               Phetchabun             5   64.600000

[392 rows x 3 columns]


/usr/local/lib/python3.12/dist-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


### Q3 Answer
Out of all cities with at least 5 readings of PM$_{2.5}$, the three cities with the most readings above 50 µg/m³ are all in India: Delhi, Mumbai, and Ahmedabad. This is consistent with data collected over 2019-2022 [[1](https://en.wikipedia.org/wiki/List_of_most-polluted_cities_by_particulate_matter_concentration)].

## 4. What are the 20 least polluted locations in the dataset by average NO${_2}$?
We'll keep only locations with at least 5 readings for reliability, and excluding all negative readings.

In [63]:
Q4_query = """
    SELECT location, country, AVG(value) as avg_no2, Count(1) as num_readings
    FROM `bigquery-public-data.openaq.global_air_quality`
    WHERE pollutant = "no2" AND value >= 0
    GROUP BY location, country
    HAVING num_readings > 5
    ORDER BY avg_no2 ASC
    LIMIT 20
    """

dry_run_query_job = client.query(Q4_query, job_config=dry_run_config)
print("This query will process {} Mb.".format(dry_run_query_job.total_bytes_processed/1e6))

This query will process 193.869959 Mb.


In [64]:
Q4_query_job = client.query(Q4_query, job_config=safe_config)
Q4_result = Q4_query_job.to_dataframe()

print(Q4_result)

                               location country   avg_no2  num_readings
0                Pietermaritzburg - CBD      ZA  0.000000           431
1                             Capricorn      ZA  0.000000           432
2            Yupparaj Wittayalai School      TH  0.000009           465
3                               CORDOBA      AR  0.000079           291
4                       Marystown/Burin      CA  0.000095            21
5   Kanchanaburi Meteorological Station      TH  0.000161           442
6                        Johnson County      US  0.000203           508
7                            South Pass      US  0.000266           492
8                  Chaco Culture Nation      US  0.000269           498
9                              Booysens      ZA  0.000290           218
10                          Lichtenburg      ZA  0.000315            46
11                          SOUTHAMPTON      CA  0.000373           415
12                                Malta      US  0.000412       

/usr/local/lib/python3.12/dist-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


## 5. For each country, how many distinct pollutants are being monitored?

In [66]:
Q5_query = """
    SELECT country, COUNT(DISTINCT pollutant) as num_pollutants
    FROM `bigquery-public-data.openaq.global_air_quality`
    GROUP BY country
    ORDER BY country
    """

dry_run_query_job = client.query(Q5_query, job_config=dry_run_config)
print("This query will process {} Mb.".format(dry_run_query_job.total_bytes_processed/1e6))

This query will process 50.796434 Mb.


In [67]:
Q5_query_job = client.query(Q5_query, job_config=safe_config)
Q5_result = Q5_query_job.to_dataframe()

print(Q5_result)

    country  num_pollutants
0        AD               5
1        AE               2
2        AF               1
3        AR               4
4        AT               6
..      ...             ...
106      UZ               2
107      VM               1
108      VN               1
109      XK               6
110      ZA               6

[111 rows x 2 columns]


/usr/local/lib/python3.12/dist-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


## 6. Which source organisations operating in at least 2 countries contribute the most readings, and what countries do they operate in?

In [91]:
Q6_query = """
    SELECT source_name, COUNT(DISTINCT country) AS num_country, COUNT(1) AS contributions
    FROM `bigquery-public-data.openaq.global_air_quality`
    GROUP BY source_name
    HAVING num_country >= 2
    ORDER BY contributions DESC
    """

dry_run_query_job = client.query(Q6_query, job_config=dry_run_config)
print("This query will process {} Mb.".format(dry_run_query_job.total_bytes_processed/1e6))

This query will process 80.883275 Mb.


In [92]:
Q6_query_job = client.query(Q6_query, job_config=safe_config)
Q6_result = Q6_query_job.to_dataframe()

print(Q6_result)

                     source_name  num_country  contributions
0                         AirNow           49        1783377
1                          DEFRA            2         184622
2         Australia - Queensland            2          38255
3               Ho Chi Minh City            2            481
4            StateAir_KuwaitCity            2            480
5           StateAir_Ulaanbaatar            2            479
6               StateAir_SanJose            2            460
7                 StateAir_Hanoi            2            332
8   StateAir_KhartoumResidential            2            290
9                 StateAir_Dhaka            2            289
10              StateAir_Rangoon            2            260
11                     Andalucia            2            159
12   Australia - South Australia            2             68
13                       Bosnia2            2             59
14                       Spartan           11             24


/usr/local/lib/python3.12/dist-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


## 7. How does average PM2.5 vary by year globally?

In [97]:
Q7_query = """
    SELECT EXTRACT(YEAR from timestamp) AS year, AVG(value) as avg_value
    FROM `bigquery-public-data.openaq.global_air_quality`
    WHERE pollutant = "pm25" AND value >= 0
    GROUP BY year
    ORDER BY year ASC
    """

dry_run_query_job = client.query(Q7_query, job_config=dry_run_config)
print("This query will process {} Mb.".format(dry_run_query_job.total_bytes_processed/1e6))

This query will process 117.931802 Mb.


In [98]:
Q7_query_job = client.query(Q7_query, job_config=safe_config)
Q7_result = Q7_query_job.to_dataframe()

print(Q7_result)

/usr/local/lib/python3.12/dist-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


    year  avg_value
0   2007  25.368333
1   2008  33.724000
2   2014  12.400000
3   2015  57.791000
4   2016  18.405681
5   2017  27.149199
6   2018  37.541292
7   2019  18.196392
8   2020  15.283042
9   2021  25.675709
10  2022  17.250298


## 8. Which UK cities have readings for at least 3 different pollutants, and what is the average value for each pollutant in those cities?

In [120]:
Q8_query = """
    WITH select_cities AS
    (
        SELECT city
        FROM `bigquery-public-data.openaq.global_air_quality`
        WHERE value >= 0 AND country = "GB"
        GROUP BY city
        HAVING COUNT(DISTINCT pollutant) >= 3
    )
    
    SELECT g.city, g.pollutant, AVG(g.value) as avg_value
    FROM `bigquery-public-data.openaq.global_air_quality` as g
    INNER JOIN select_cities as c
        ON g.city = c.city
    WHERE g.value >= 0
    GROUP BY g.city, g.pollutant
    ORDER BY g.city, g.pollutant
    """

dry_run_query_job = client.query(Q8_query, job_config=dry_run_config)
print("This query will process {} Mb.".format(dry_run_query_job.total_bytes_processed/1e6))

This query will process 168.989454 Mb.


In [121]:
Q8_query_job = client.query(Q8_query, job_config=safe_config)
Q8_result = Q8_query_job.to_dataframe()

print(Q8_result)

         city pollutant  avg_value
0    Aberdeen       no2  22.516227
1    Aberdeen        o3  22.000000
2    Aberdeen      pm10   6.000000
3    Aberdeen      pm25   2.691602
4        Adur       no2   8.900000
..        ...       ...        ...
372   Wrexham      pm25   8.737931
373   Wrexham       so2   3.175097
374      York       no2  11.292369
375      York      pm10  16.379822
376      York      pm25   7.148368

[377 rows x 3 columns]


/usr/local/lib/python3.12/dist-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


## 9. For each country, what percentage of readings have a negative value (i.e. are likely erroneous)?

In [132]:
Q9_query = """

    WITH country_total AS (
        SELECT country, COUNT(*) as total
        FROM `bigquery-public-data.openaq.global_air_quality`
        GROUP BY country
    ),

    country_neg AS (
        SELECT country, COUNT(*) as neg
        FROM `bigquery-public-data.openaq.global_air_quality`
        WHERE value < 0
        GROUP BY country
    )
    
    SELECT t.country, n.neg * 100 / t.total as perc_invalid
    FROM country_total as t
        INNER JOIN country_neg as n ON t.country = n.country
    ORDER BY country
    """

dry_run_query_job = client.query(Q9_query, job_config=dry_run_config)
print("This query will process {} Mb.".format(dry_run_query_job.total_bytes_processed/1e6))

This query will process 67.135368 Mb.


In [133]:
Q9_query_job = client.query(Q9_query, job_config=safe_config)
Q9_result = Q9_query_job.to_dataframe()

print(Q9_result)

   country  perc_invalid
0       AE      1.986434
1       AF     25.000000
2       AT      0.193886
3       AU      1.822303
4       BA      0.564972
..     ...           ...
68      UZ      3.435115
69      VM      0.469484
70      VN     39.901478
71      XK     21.558872
72      ZA      2.017501

[73 rows x 2 columns]


/usr/local/lib/python3.12/dist-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


## 10. Which locations have readings for both PM2.5 and NO2, and how do their average values compare side by side?

In [141]:
Q10_query = """

    WITH select_countries_pm25 AS (
        SELECT country, AVG(value) as avg_pm25
        FROM `bigquery-public-data.openaq.global_air_quality`
        WHERE pollutant = "pm25" AND value >= 0
        GROUP BY country
    ),

    select_countries_no2 AS (
        SELECT country, AVG(value) as avg_no2
        FROM `bigquery-public-data.openaq.global_air_quality`
        WHERE pollutant = "no2" AND value >= 0
        GROUP BY country
    )
    
    SELECT p.country, p.avg_pm25, n.avg_no2
    FROM select_countries_pm25 as p
        INNER JOIN select_countries_no2 as n ON p.country = n.country
    ORDER BY country
    """

dry_run_query_job = client.query(Q10_query, job_config=dry_run_config)
print("This query will process {} Mb.".format(dry_run_query_job.total_bytes_processed/1e6))

This query will process 95.553346 Mb.


In [142]:
Q10_query_job = client.query(Q10_query, job_config=safe_config)
Q10_result = Q10_query_job.to_dataframe()

print(Q10_result)

   country   avg_pm25    avg_no2
0       AR  13.066667   0.007685
1       AT   8.945114  12.368899
2       AU   5.264174   0.007351
3       BA  18.623077  21.280556
4       BE  11.458534  17.063972
5       BG  15.211898  15.778995
6       BR  14.346302  24.827874
7       CA   4.638089   0.003814
8       CH  23.087883  13.988209
9       CL  41.929945  21.611687
10      CN  25.989270  18.604470
11      CO  17.690127   0.017337
12      CZ  13.611963  14.396218
13      DE   9.756537  14.365823
14      DK   8.012322   9.160385
15      EC   0.000000  37.673198
16      EE   7.312107   6.102172
17      ES   9.068833  11.369018
18      FI   3.575821   7.042605
19      FR   9.060785  13.268518
20      GB   8.757788  17.569113
21      GR  13.399642  31.180907
22      HK  12.137010  33.334477
23      HR  11.245843  15.320789
24      HU  12.614601  18.734257
25      IE   7.568329  11.301704
26      IL  18.850000   0.018995
27      IN  65.861893  26.162622
28      IS   5.166143   4.251834
29      IT

/usr/local/lib/python3.12/dist-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(
